# Notebook 04 — Evaluation
**DSP501 — Speaker Identification**

This notebook produces all figures for the IEEE report:
- Metrics table (Accuracy, F1, Precision, Recall)
- Confusion matrix for each pipeline
- ROC curves (one-vs-rest)
- Bar chart comparison A1 vs B1
- Statistical test summary

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from evaluation import (
    compute_metrics, compute_ci, paired_ttest,
    plot_confusion_matrix, plot_roc_curve, plot_comparison_table
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load data and models

In [ ]:
X_raw  = np.load('../features/features_mfcc_raw.npy')
X_filt = np.load('../features/features_mfcc_filt.npy')
y      = np.load('../features/labels.npy')

model_a = joblib.load('../models/svm_pipeline_a.pkl')
model_b = joblib.load('../models/svm_pipeline_b.pkl')

with open('../results.json') as f:
    results = json.load(f)

print('Loaded successfully.')

## 2. Metrics Table

In [ ]:
y_pred_a = model_a.predict(X_raw)
y_pred_b = model_b.predict(X_filt)

metrics_a = compute_metrics(y, y_pred_a)
metrics_b = compute_metrics(y, y_pred_b)

df = pd.DataFrame([metrics_a, metrics_b],
                   index=['A1 — Raw', 'B1 — Filtered'])
df = df.round(4)
print(df.to_string())
df

## 3. Confusion Matrices

In [ ]:
speaker_labels = [f'Spk {i}' for i in sorted(set(y.tolist()))]

plot_confusion_matrix(y, y_pred_a, labels=speaker_labels,
                      title='Pipeline A — Raw',
                      save_path='../figures/confusion_matrix_a1.png')

In [ ]:
plot_confusion_matrix(y, y_pred_b, labels=speaker_labels,
                      title='Pipeline B — Filtered',
                      save_path='../figures/confusion_matrix_b1.png')

## 4. ROC Curves

In [ ]:
plot_roc_curve(model_a, X_raw, y,
               save_path='../figures/roc_curve_a1.png')

In [ ]:
plot_roc_curve(model_b, X_filt, y,
               save_path='../figures/roc_curve_b1.png')

## 5. Pipeline Comparison Bar Chart

In [ ]:
plot_comparison_table(results_path='../results.json',
                      save_path='../figures/comparison_bar.png')

## 6. Statistical Test Summary

In [ ]:
test = results['statistical_tests']['SVM_A_vs_B']
exp  = results['experiments']

print('=== Statistical Test: Paired t-test (A1 vs B1) ===')
print(f"  t-statistic : {test['t_stat']:.4f}")
print(f"  p-value     : {test['p_value']:.4f}")
print()
print('=== Accuracy Summary ===')
for name, e in exp.items():
    acc = e['accuracy']
    print(f"  {name:20s}  mean={acc['mean']:.4f}  std={acc['std']:.4f}  "
          f"95%CI=[{acc['ci_95'][0]:.4f}, {acc['ci_95'][1]:.4f}]")
print()
if test['p_value'] < 0.05:
    print('Conclusion: FIR filtering SIGNIFICANTLY improves accuracy (p < 0.05)')
else:
    print('Conclusion: No statistically significant improvement (p >= 0.05)')